# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #4 — "The Freshness Multiplier" (361+ day bucket, 283:1 growth-to-decline ratio):
Where the label comes from: growth vs. decline is defined by the 30-day trend rule (>10% change vs. prior 30 days), same as my own is_churned label.
Does the validation design carry the claim? The paper itself flags this honestly — the 283:1 ratio comes from only 1 declining page in that bucket. A ratio built on a single denominator observation is extremely fragile; one different page landing in "declining" would collapse the ratio to roughly 140:1, and a few more could erase it entirely. The paper's own caveat that this shouldn't be treated as a "headline multiplier" is the right call, and I'd extend it further: I would not report this specific number in any client-facing summary without also reporting the raw counts (283 vs 1) right next to it.

Finding — ML Appendix Feature Importance (Random Forest predicting Health Score, Avg Position = 43% importance):
Where the label comes from: Health Score itself is a composite built from impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts) — so Average Position is literally one of the four ingredients used to build the score the model is predicting.
Does the validation design carry the claim? This is the same leakage pattern I found in my own ML-05 audit with trend_pct. The paper is transparent about this ("importance is descriptive rather than causal"), which is the right disclosure — but it means this chart can't be read as "position drives health" causally; it's closer to a sanity check that the model rediscovered its own scoring formula.

In [5]:
print("Two findings audited from flyrank-seo-research-march-2026.pdf:")
print("1. Finding #4 (Freshness Multiplier, 361+ bucket) - flagged as unstable due to n=1 in denominator")
print("2. ML Appendix Feature Importance (Health Score RF) - flagged as circular, target partly built from top features")

Two findings audited from flyrank-seo-research-march-2026.pdf:
1. Finding #4 (Freshness Multiplier, 361+ bucket) - flagged as unstable due to n=1 in denominator
2. ML Appendix Feature Importance (Health Score RF) - flagged as circular, target partly built from top features


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-confirming my Week-5 client-holdout split result, comparing it against what a naive random row split would have shown — to demonstrate why the honest split matters.

In [6]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)

numeric_features = ["impressions_90d","search_volume","competition","cpc","word_count","char_count",
                     "days_with_impressions","days_with_sessions","content_age_days",
                     "days_since_last_update","ctr","avg_position","engagement_rate",
                     "scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
                         "freshness_tier","word_count_tier","impression_tier","position_tier"]
X_numeric = df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_categorical = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_churned"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df["client_id"]))
model_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model_honest.fit(X.iloc[tr_idx], y.iloc[tr_idx])
honest_scores = model_honest.predict_proba(X.iloc[te_idx])[:, 1]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_naive = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model_naive.fit(X_tr2, y_tr2)
naive_scores = model_naive.predict_proba(X_te2)[:, 1]

for k in (20, 50):
    print(f"Precision@{k} - client-holdout (honest): {precision_at_k(honest_scores, y.iloc[te_idx].values, k):.3f}   |   random row split (naive): {precision_at_k(naive_scores, y_te2.values, k):.3f}")

Precision@20 - client-holdout (honest): 0.700   |   random row split (naive): 0.950
Precision@50 - client-holdout (honest): 0.680   |   random row split (naive): 0.940


The gap is striking: naive precision (0.950/0.940) is dramatically inflated compared to the honest client-holdout number (0.700/0.680) — roughly 25-35% higher. This confirms the split choice isn't a technicality; a random split would have let me report a model that looks far stronger than it actually is, simply because it partially memorized client-specific patterns rather than learning transferable signal.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the same leakage hunt from ML-05 on my final feature set, to confirm nothing changed after adding the client-holdout split and error analysis in ML-08.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

clean_auc = roc_auc_score(y, DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X, y).predict_proba(X)[:,1])
X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"].fillna(0)
leaky_auc = roc_auc_score(y, DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y).predict_proba(X_leaky)[:,1])
print(f"Clean AUC: {clean_auc:.3f}  |  Leaky (trend_pct) AUC: {leaky_auc:.3f}")
print("Confirmed: final feature set (X) does not include trend_pct or any label-derived column.")

Clean AUC: 0.673  |  Leaky (trend_pct) AUC: 1.000
Confirmed: final feature set (X) does not include trend_pct or any label-derived column.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence, originally: "A model beats a hand-written rule at ranking pages by churn risk."
Rewritten, safely: "In this dataset and under a client-holdout split, a Random Forest model achieved comparable but not consistently superior Precision@20/50 versus a carefully audited hand rule — the two approaches are directionally complementary rather than one clearly replacing the other, and this has not been validated against the full data warehouse or a different time period."

In [8]:
print("Bold claim rewritten in careful language - see markdown above.")
print("Key hedges added: 'in this dataset', 'under this split', 'not validated on full warehouse'.")

Bold claim rewritten in careful language - see markdown above.
Key hedges added: 'in this dataset', 'under this split', 'not validated on full warehouse'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.